# H5: checkpoint averaging of the locked RAMP baseline

Averages the saved epoch 60/70/80 weights of `ramp_scratch_l5_s{0,1}` (pulled from the Hub), evaluates the average with the unchanged
fixed-eps AutoAttack protocol (1000 test points) and compares with the baseline. Seed 0 on GPU 0, seed 1 on GPU 1, about 35 session-minutes.
Needs the `HF_TOKEN` secret, GPU T4 x2, Internet on.

In [ ]:
import os
os.environ['HF_REPO'] = 'matokebryan/aat-checkpoints'
os.environ['NTFY_TOPIC'] = 'aat-matoke-7q4r9x'   # subscribe to this in the ntfy app
os.environ['TIME_BUDGET_H'] = '10.5'             # training stops cleanly after this and pushes its checkpoint
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
%cd /kaggle/working
!git clone -q -b matt/funny-planck-3j7md5 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT

def stage(name, cmd):
    """Run a stage, stream its output, push a phone notification, and stop the notebook if it fails."""
    get_ipython().system(f'python scripts/stage.py {name} "{cmd}"')
    if _exit_code != 0:
        raise SystemExit(f'stage {name} failed (exit {_exit_code}): see the notification or logs/{name}.txt on Hugging Face')

stage('install', 'bash scripts/ramp_official.sh install')   # dependencies + full import check, before anything else
stage('data', 'bash scripts/ramp_official.sh data')         # CIFAR-10 (cached on Hugging Face after the first run)

stage('rampavg', 'bash scripts/ramp_official.sh avg2 && '
      'python scripts/compare_runs.py --names RAMPavg RAMP '
      '--a \\"runs_official/rampavg_s*/eval_autoattack.json\\" '
      '--b \\"baselines/results/ramp_scratch_l5_s*_eval_autoattack.json\\"')

In [ ]:
## Average and evaluate